In [ ]:
# NORTHSTAR -- Tower 15 App Makers: Deep QA for Solace Browser
# DNA: platform(thriving) = sdk(intuitive) x docs(complete) x store(fair) x feedback(fast) x templates(inspiring) x monetization(transparent) x community(growing)
# This notebook extends 05-app-makers-qa.ipynb with deeper probes on store, SDK, and monetization
import os, sys, re, json, hashlib, ast
from pathlib import Path
from datetime import datetime

NORTHSTAR = "app-makers-t15-deep-qa"
NOTEBOOK_PATH = "notebooks/qa/13-app-makers-t15-qa.ipynb"
TOWER = 15
TOWER_NAME = "Tower of App Makers"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
DATA = BROWSER_ROOT / "data" / "default"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

print(f"Tower {TOWER}: {TOWER_NAME} -- Deep QA")
print(f"Target: solace-browser")
print(f"Browser root: {BROWSER_ROOT}")
assert BROWSER_ROOT.exists(), f"Browser root not found: {BROWSER_ROOT}"

In [ ]:
# Cell 1: Bootstrap -- imports and helpers

def read_file(path):
    """Read file content safely."""
    p = Path(path)
    if p.exists():
        return p.read_text(encoding='utf-8', errors='replace')
    return ""

def count_pattern(path, pattern, flags=0):
    """Count regex matches in a file."""
    content = read_file(path)
    return len(re.findall(pattern, content, flags))

def file_exists_with_content(path, min_bytes=50):
    """Check file exists and has meaningful content."""
    p = Path(path)
    return p.exists() and p.stat().st_size >= min_bytes

print("Bootstrap complete")

In [ ]:
# F1 -- First-Time App Builder (persona: beginner developer)
# Q: Can a new dev go from zero to published app in <30 min using docs?
# Q: Is there a create-app wizard or template?

# Probe: create-app page exists and has form/wizard elements
create_app_html = read_file(WEB / "create-app.html")
has_create_page = len(create_app_html) > 100
record("F1-P1", "create-app.html page exists with content", has_create_page,
       f"{len(create_app_html)} bytes")

# Probe: create-app has form controls for app creation
has_form = bool(re.search(r'<form|<input|<select|<textarea', create_app_html, re.IGNORECASE))
record("F1-P2", "create-app has form controls", has_form)

# Probe: app store page exists
store_html = read_file(WEB / "app-store.html")
has_store = len(store_html) > 100
record("F1-P3", "app-store.html page exists", has_store, f"{len(store_html)} bytes")

# Probe: quick-start docs exist and mention apps
quick_start = read_file(WEB / "docs" / "quick-start.html")
mentions_app = bool(re.search(r'app|recipe|skill', quick_start, re.IGNORECASE))
record("F1-P4", "Quick-start docs mention apps/recipes", mentions_app,
       f"{len(quick_start)} bytes in quick-start")

In [ ]:
# F2 -- Recipe Author (persona: recipe creator)
# Q: Is there a recipe authoring tool that scaffolds JSON schema?
# Q: Can recipe authors test in sandbox?

# Probe: recipe parser validates schema
parser_path = SRC / "recipes" / "recipe_parser.py"
parser_code = read_file(parser_path)
has_validation = bool(re.search(r'validate|schema|required.*field|jsonschema', parser_code, re.IGNORECASE))
record("F2-P1", "Recipe parser has schema validation", has_validation)

# Probe: recipe compiler exists with IR (intermediate representation)
compiler_path = SRC / "recipes" / "recipe_compiler.py"
compiler_code = read_file(compiler_path)
has_ir = bool(re.search(r'compile|IR|intermediate|transform', compiler_code, re.IGNORECASE))
record("F2-P2", "Recipe compiler generates IR", has_ir,
       f"{len(compiler_code)} bytes")

# Probe: recipe test files exist
recipe_tests = list(TESTS.glob("test_recipe_*.py"))
record("F2-P3", "Recipe test suite exists", len(recipe_tests) >= 3,
       f"{len(recipe_tests)} recipe test files")

# Probe: recipes directory has JSON recipe files
recipe_files = list(DATA.rglob("*.json")) if DATA.exists() else []
recipe_jsons = [f for f in recipe_files if 'recipe' in str(f).lower()]
record("F2-P4", "Recipe JSON files exist in data/default", len(recipe_jsons) > 0,
       f"{len(recipe_jsons)} recipe files found")

In [ ]:
# F3 -- OAuth3 Integrator (persona: app dev integrating OAuth3)
# Q: Does the app creation flow explain OAuth3 scopes?
# Q: Is there scope documentation?

# Probe: OAuth3 scopes module defines available scopes
scopes_path = SRC / "oauth3" / "scopes.py"
scopes_code = read_file(scopes_path)
scope_defs = re.findall(r'["\']([a-z_]+:[a-z_]+)["\']', scopes_code)
record("F3-P1", "OAuth3 scopes are defined with namespace:action format",
       len(scope_defs) >= 3, f"{len(scope_defs)} scope definitions: {scope_defs[:5]}")

# Probe: consent UI exists for showing scopes to users
consent_path = SRC / "oauth3" / "consent_ui.py"
consent_code = read_file(consent_path)
has_consent = bool(re.search(r'consent|permission|approve|deny|scope', consent_code, re.IGNORECASE))
record("F3-P2", "Consent UI module exists with scope display", has_consent)

# Probe: OAuth3 docs page exists
oauth3_docs = read_file(WEB / "docs" / "oauth3.html")
has_oauth_docs = len(oauth3_docs) > 100
record("F3-P3", "OAuth3 documentation page exists", has_oauth_docs,
       f"{len(oauth3_docs)} bytes")

# Probe: app-detail page shows scope info
app_detail = read_file(WEB / "app-detail.html")
shows_scopes = bool(re.search(r'scope|permission|access', app_detail, re.IGNORECASE))
record("F3-P4", "App detail page references scopes/permissions", shows_scopes)

In [ ]:
# F4 -- Store Publisher (persona: app submitter)
# Q: Is submission documented with clear acceptance criteria?
# Q: Is there an automated pre-review?

# Probe: app store backend exists
store_backend = SRC / "app_store" / "backend.py"
store_code = read_file(store_backend)
has_submit = bool(re.search(r'submit|publish|review|approve', store_code, re.IGNORECASE))
record("F4-P1", "App store backend has submission logic", has_submit,
       f"{len(store_code)} bytes")

# Probe: store client exists for API interaction
store_client = SRC / "store_client" / "store_client.py"
client_code = read_file(store_client)
has_client = len(client_code) > 100
record("F4-P2", "Store client module exists", has_client,
       f"{len(client_code)} bytes")

# Probe: store tests exist
store_tests = list(TESTS.glob("test_store_*.py"))
record("F4-P3", "Store integration tests exist", len(store_tests) >= 1,
       f"{len(store_tests)} store test files")

# Probe: app runner handles installed apps
runner_path = SRC / "apps" / "runner.py"
runner_code = read_file(runner_path)
has_runner = bool(re.search(r'run|execute|launch|install', runner_code, re.IGNORECASE))
record("F4-P4", "App runner handles app execution", has_runner)

In [ ]:
# F7 -- Theme Designer (persona: visual customization)
# Q: Are there theme CSS files that app makers can reference?
# Q: Is there a style guide page?

# Probe: multiple theme CSS files exist
theme_dir = WEB / "css" / "themes"
theme_files = list(theme_dir.glob("*.css")) if theme_dir.exists() else []
record("F7-P1", "Multiple theme CSS files exist", len(theme_files) >= 2,
       f"{len(theme_files)} themes: {[f.name for f in theme_files]}")

# Probe: CSS custom properties (design tokens) exist
site_css = read_file(WEB / "css" / "site.css")
custom_props = re.findall(r'--[a-z][a-z0-9-]+', site_css)
unique_props = set(custom_props)
record("F7-P2", "CSS design tokens (custom properties) defined",
       len(unique_props) >= 10, f"{len(unique_props)} unique custom properties")

# Probe: style guide page exists
styleguide_html = read_file(WEB / "style-guide.html")
has_styleguide = len(styleguide_html) > 100
record("F7-P3", "Style guide HTML page exists", has_styleguide,
       f"{len(styleguide_html)} bytes")

# Probe: theme.js handles theme switching
theme_js = read_file(WEB / "js" / "theme.js")
has_theme_switch = bool(re.search(r'setTheme|switchTheme|toggle.*theme|theme.*select', theme_js, re.IGNORECASE))
record("F7-P4", "theme.js handles theme switching", has_theme_switch)

In [ ]:
# F9 -- Documentation Writer + F12 -- i18n App Maker (NEGATIVE SPACE)
# P16: Check what's MISSING

# Probe: API documentation (openapi.yaml) exists
openapi = SRC / "api" / "openapi.yaml"
has_openapi = file_exists_with_content(openapi, 200)
record("F9-P1", "OpenAPI spec exists for API documentation", has_openapi)

# Probe: MCP docs exist
mcp_docs = read_file(WEB / "docs" / "mcp.html")
has_mcp_docs = len(mcp_docs) > 100
record("F9-P2", "MCP documentation page exists", has_mcp_docs)

# Negative space: Are there HTML pages missing i18n data-i18n attributes?
html_files = list(WEB.glob("*.html"))
pages_without_i18n = []
for hf in html_files:
    if hf.name.startswith("partials"):
        continue
    content = hf.read_text(encoding='utf-8', errors='replace')
    has_i18n = bool(re.search(r'data-i18n|lang=|i18n', content, re.IGNORECASE))
    if not has_i18n:
        pages_without_i18n.append(hf.name)
record("F12-P1", "All pages have i18n attributes",
       len(pages_without_i18n) == 0,
       f"Missing i18n: {pages_without_i18n}" if pages_without_i18n else "all pages have i18n")

# Negative space: Is there a CHANGELOG or release notes file?
has_changelog = (BROWSER_ROOT / "CHANGELOG.md").exists() or (BROWSER_ROOT / "RELEASES.md").exists()
record("F9-P3", "CHANGELOG or release notes exist", has_changelog)

In [ ]:
# F48 -- Integration: App Maker Pipeline Coherence
# Q: Does create-app -> store -> install -> run work as a pipeline?

# Probe: companion module bridges app ecosystem
companion_apps = read_file(SRC / "companion" / "apps.py")
has_companion = bool(re.search(r'install|list|run|uninstall', companion_apps, re.IGNORECASE))
record("F48-P1", "Companion apps module has install/run lifecycle", has_companion)

# Probe: cross-app orchestrator exists
cross_orch = read_file(SRC / "cross_app" / "orchestrator.py")
has_orch = len(cross_orch) > 100
record("F48-P2", "Cross-app orchestrator exists", has_orch,
       f"{len(cross_orch)} bytes")

# Probe: metrics tracking for recipe performance
metrics_path = SRC / "recipes" / "metrics.py"
metrics_code = read_file(metrics_path)
has_metrics = bool(re.search(r'hit_rate|cost|latency|success', metrics_code, re.IGNORECASE))
record("F48-P3", "Recipe metrics track hit rate and cost", has_metrics)

# Probe: budget gates enforce cost limits
budget_code = read_file(SRC / "budget_gates.py")
has_budget = bool(re.search(r'budget|limit|cap|exceed', budget_code, re.IGNORECASE))
record("F48-P4", "Budget gates enforce cost limits for apps", has_budget)

In [ ]:
# EVIDENCE SUMMARY -- Tower 15 App Makers Deep QA
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"

summary = {
    "surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total, "passed": passed, "failed": failed,
    "score": score, "grade": grade, "finding_rate": finding_rate,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME} -- DEEP QA")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))